<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

# Subtask 1: Dimensional Aspect Sentiment Regression (DimASR)

-----

## Starter Notebook
Leveraging Pretrained Language Models for Dimensional Sentiment Regression


## Introduction:

You are welcome to participate in our SemEval Shared Task!

In this starter notebook, we will take you through the process of fine-tuning a pre-trained language model on a sample data to build a sentiment regressor. The notebook was adapted from a Hugginface implementation for such tasks.

### Outline:

- Installation and importation of necessary libraries
Setting up the project parameters.
Running training and evaluation
Before you start:

- It is strongly advised that you use a GPU to speed up training. To do this, go to the "Runtime" menu in Colab, select "Change runtime type" and then in the popup menu, choose "GPU" in the "Hardware accelerator" box.

### NB:

The codes in this notebook are provided to familiarize yourselves with fine-tuning language models for sentiment regression. You may extend and (or) modify as appropriate to obtain competitive performances.

### Languages and Domains:
#### Track A: Subtask 1
- eng_restaurant
- eng_laptop
- jpn_hotel
- jpn_finance
- rus_restaurant
- tat_restaurant
- ukr_restaurant
- zho_restaurant
- zho_laptop
#### Track B: Subtask 1
- deu-stance
- eng-stance
- hau-stance
- kin-stance
- swa-stance
- twi-stance



In [ ]:
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DebertaV2TokenizerFast

from scipy.stats import pearsonr
from tqdm import tqdm
import math
import re
import requests


def load_jsonl(filepath: str) -> List[Dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_jsonl_url(url: str) -> List[Dict]:
    resp = requests.get(url)
    resp.raise_for_status()
    return [json.loads(line) for line in resp.text.splitlines()]

### First, visit the [DimABSA2006](https://github.com/DimABSA/DimABSA2026) repository, check the task-dataset.

### Step 1: Load the competition data

- Read JSONL files (train/dev/predict) into Colab.  
- Train files contain Valence–Arousal (VA) labels.  
- Predict files have no VA labels.  
- This script:
  1. Loads the JSONL data.
  2. Splits 10% of train data as dev set.
  3. Converts JSONL into DataFrames (ID, Text, Aspect, Valence, Arousal).
  4. Prints the first few rows for checking.


In [ ]:
#task config
subtask = "subtask_1"
task = "task1"
lang = "eng"
domain = "restaurant"

train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl"
predict_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"

train_raw = load_jsonl_url(train_url)
predict_raw = load_jsonl_url(predict_url)

In [ ]:


def jsonl_to_df(data):
    if 'Quadruplet' in data[0]:
        df = pd.json_normalize(data, 'Quadruplet', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA', 'Category', 'Opinion'])
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')

    elif 'Triplet' in data[0]:
        df = pd.json_normalize(data, 'Triplet', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA', 'Opinion'])
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')

    elif 'Aspect_VA' in data[0]:
        df = pd.json_normalize(data, 'Aspect_VA', ['ID', 'Text'])
        df = df.rename(columns={df.columns[0]: "Aspect"})
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')

    elif 'Aspect' in data[0]:
        df = pd.json_normalize(data, 'Aspect', ['ID', 'Text'])
        df = df.rename(columns={df.columns[0]: "Aspect"})
        df['Valence'] = 0  # default value
        df['Arousal'] = 0  # default value

    else:
        raise ValueError("Invalid format: must include 'Quadruplet' or 'Triplet' or 'Aspect'")

    return df

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)

In [ ]:
print("\n--- Training Data Sample (With Labels) ---")
print(train_df[['ID', 'Aspect', 'Valence', 'Arousal']].head())

print("\n--- Prediction Data Sample (Without Labels) ---")
print(predict_df[['ID', 'Aspect', 'Valence', 'Arousal']].head())


--- Training Data Sample (With Labels) ---
                          ID      Aspect  Valence  Arousal
481     rest16_quad_test_127    winelist     8.25     8.38
2520   rest16_quad_train_814        veal     5.67     5.33
3067  rest16_quad_train_1157    bathroom     3.33     5.83
667     rest16_quad_test_222  crab cakes     6.88     6.50
2462   rest16_quad_train_781        NULL     3.17     7.00

--- Prediction Data Sample (Without Labels) ---
                       ID      Aspect  Valence  Arousal
0  rest26_aspect_va_dev_1  diner food        0        0
1  rest26_aspect_va_dev_1   breakfast        0        0
2  rest26_aspect_va_dev_2        food        0        0
3  rest26_aspect_va_dev_2      drinks        0        0
4  rest26_aspect_va_dev_2     service        0        0


### Display the dataframe

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### subtask_1_eng_restaurant train_df

,Aspect,ID,Text,Valence,Arousal
481,winelist,rest16_quad_test_127,seattle ' s best winelist,8.25,8.38
2520,veal,rest16_quad_train_814,"the restaurant has a family feel , not least w...",5.67,5.33
3067,bathroom,rest16_quad_train_1157,"service ok but unfriendly , filthy bathroom .",3.33,5.83
667,crab cakes,rest16_quad_test_222,best crab cakes in town,6.88,6.50
2462,NULL,rest16_quad_train_781,they are not helpful in the least and will giv...,3.17,7.00


### subtask_1_eng_restaurant dev_df

,Aspect,ID,Text,Valence,Arousal
172,NULL,rest16_quad_dev_118,way below average,2.67,7.00
351,pizza place,rest16_quad_test_53,mama mia – i live in the neighborhood and feel...,7.88,8.00
3132,cocktail with citrus vodka and lemon and lime ...,rest16_quad_train_1191,the have a great cocktail with citrus vodka an...,7.88,8.12
1993,place,rest16_quad_train_499,not a great place for family or general dining .,3.00,6.75
366,food,rest16_quad_test_64,"the food is great , the bartenders go that ext...",7.67,7.50


### subtask_1_eng_restaurant predict_df

,Aspect,ID,Text,Valence,Arousal
0,diner food,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,0,0
1,breakfast,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,0,0
2,food,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,0,0
3,drinks,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,0,0
4,service,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,0,0


### Model:
This Starter Notebook uses the bert-base-multilingual-cased pretrained model, developed by Google. The model was trained with a masked language modeling (MLM) objective on the top 104 languages with the largest Wikipedia presence. You can find the model here: https://huggingface.co/google-bert/bert-base-multilingual-cased

If your target language is not included in the common set supported by this model, you can search for a more suitable model on Hugging Face: https://huggingface.co/models

another transformer models you can try:
1. roberta-large
2. roberta-base
3. bert-base-uncased

more models please visit [huggingface](https://huggingface.co/models)

In [ ]:
# Models
model_candidates = [
    "bert-base-uncased",
    "bert-base-cased",
    "roberta-base",
    "roberta-large",
    "microsoft/deberta-v3-base"
]

tokenizers = {}

for m_name in model_candidates:
    print(f"Initializing tokenizer for: {m_name}")
    if "deberta-v3" in m_name:
        tokenizers[m_name] = DebertaV2TokenizerFast.from_pretrained(m_name)
    else:
        tokenizers[m_name] = AutoTokenizer.from_pretrained(m_name)

class MultiModelVADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.sentences = dataframe["Text"].tolist()
        self.aspects = dataframe["Aspect"].tolist()
        self.labels = dataframe[["Valence", "Arousal"]].values.astype(float)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # join Aspect and Text with a separator for better context
        text = f"{self.aspects[idx]}: {self.sentences[idx]}"
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

Initializing tokenizer for: bert-base-uncased


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Initializing tokenizer for: microsoft/deberta-v3-base


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


### Step 2: Build Dataset and DataLoader

- Define a custom `VADataset` class for PyTorch:
  - Joins Aspect + Text into a single input string.
  - Uses BERT tokenizer to create `input_ids` and `attention_mask`.
  - Returns `[Valence, Arousal]` labels as float tensor.
- Convert the processed DataFrames into PyTorch `Dataset` objects.
- Wrap them with `DataLoader` for mini-batch training and evaluation.

In [ ]:
all_loaders = {}

base_batch_size = 32
large_batch_size = 8

for model_name, tokenizer in tokenizers.items():
    print(f"DataLoaders for: {model_name}...")

    current_batch_size = large_batch_size if "large" in model_name else base_batch_size

    # 1. Initialize the Datasets for this specific model/tokenizer
    train_ds = MultiModelVADataset(train_df, tokenizer)
    dev_ds = MultiModelVADataset(dev_df, tokenizer)
    predict_ds = MultiModelVADataset(predict_df, tokenizer)

    # 2. Wrap them in DataLoaders
    all_loaders[model_name] = {
        'train': DataLoader(train_ds, batch_size=current_batch_size, shuffle=True),
        'dev': DataLoader(dev_ds, batch_size=current_batch_size, shuffle=False),
        'predict': DataLoader(predict_ds, batch_size=current_batch_size, shuffle=False)
    }

print("\n DataLoaders are ready for experiments!")

DataLoaders for: bert-base-uncased...
DataLoaders for: bert-base-cased...
DataLoaders for: roberta-base...
DataLoaders for: roberta-large...
DataLoaders for: microsoft/deberta-v3-base...

 DataLoaders are ready for experiments!


### Step 3: Build and Train TransformerVARegressor

- Define **`TransformerVARegressor`**:  
  - Uses pretrained Transformer (e.g. BERT) as backbone.  
  - Adds dropout and linear layer to predict **Valence** and **Arousal**.  

- Implement helper methods:  
  - `train_epoch`: one training pass with optimizer and loss.  
  - `eval_epoch`: validation pass without gradient updates.  

- Set training parameters:  
  - `lr = 1e-5`, `epochs = 5`, `loss_fn = MSELoss`.  

- Run training loop:  
  - For each epoch, print training and validation loss to monitor progress.


In [ ]:
import torch.nn as nn
from transformers import AutoModel

class TransformerVARegressor(nn.Module):
    def __init__(self, model_name):
        super(TransformerVARegressor, self).__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Linear(self.transformer.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)
        return self.regressor(pooled_output)

In [ ]:
def train_epoch(model, data_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(data_loader)

def eval_epoch(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

    return total_loss / len(data_loader)

In [ ]:
import gc
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
performance_history = []

for model_name in model_candidates:
    print(f"\n{'-'*30}")
    print(f" TRAINING MODEL: {model_name}")
    print(f"{'-'*30}")

    # Initialize Model, Optimizer, and Loss
    model = TransformerVARegressor(model_name).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn = torch.nn.MSELoss()

    # Get the specific loaders for this model
    train_loader = all_loaders[model_name]['train']
    dev_loader = all_loaders[model_name]['dev']

    best_val_loss = float('inf')
    epochs = 5

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss = eval_epoch(model, dev_loader, loss_fn, device)

        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss

    # Store results
    performance_history.append({
        'Model': model_name,
        'Best Val Loss (MSE)': best_val_loss
    })

    del model
    del optimizer
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {model_name}. GPU Memory Cleared.")

print("\n\n ALL MODELS TRAINED!")

Using device: cuda

------------------------------
 TRAINING MODEL: bert-base-uncased
------------------------------


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1: Train Loss: 5.9370 | Val Loss: 2.2503
Epoch 2: Train Loss: 1.4849 | Val Loss: 1.2597
Epoch 3: Train Loss: 0.9323 | Val Loss: 1.0786
Epoch 4: Train Loss: 0.7348 | Val Loss: 0.9016
Epoch 5: Train Loss: 0.6225 | Val Loss: 1.0892
Finished bert-base-uncased. GPU Memory Cleared.

------------------------------
 TRAINING MODEL: bert-base-cased
------------------------------


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch 1: Train Loss: 5.2450 | Val Loss: 1.7205
Epoch 2: Train Loss: 1.1853 | Val Loss: 1.1907
Epoch 3: Train Loss: 0.8290 | Val Loss: 0.9984
Epoch 4: Train Loss: 0.6280 | Val Loss: 1.0102
Epoch 5: Train Loss: 0.5209 | Val Loss: 0.9147
Finished bert-base-cased. GPU Memory Cleared.

------------------------------
 TRAINING MODEL: roberta-base
------------------------------


...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1: Train Loss: 3.7160 | Val Loss: 2.1998
Epoch 2: Train Loss: 1.4625 | Val Loss: 1.3932
Epoch 3: Train Loss: 0.8638 | Val Loss: 0.7181
Epoch 4: Train Loss: 0.6922 | Val Loss: 0.9653
Epoch 5: Train Loss: 0.5689 | Val Loss: 0.7809
Finished roberta-large. GPU Memory Cleared.

------------------------------
 TRAINING MODEL: microsoft/deberta-v3-base
------------------------------


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch 1: Train Loss: 9.5640 | Val Loss: 2.7427
Epoch 2: Train Loss: 1.4882 | Val Loss: 1.6963
Epoch 3: Train Loss: 1.0713 | Val Loss: 1.1573
Epoch 4: Train Loss: 0.8248 | Val Loss: 1.0359
Epoch 5: Train Loss: 0.7535 | Val Loss: 1.1846
Finished microsoft/deberta-v3-base. GPU Memory Cleared.


 ALL MODELS TRAINED!


In [ ]:
# Table to compare results
leaderboard = pd.DataFrame(performance_history)
leaderboard = leaderboard.sort_values(by='Best Val Loss (MSE)', ascending=True)

print("--- MODEL LEADERBOARD ---")
print(leaderboard)

--- MODEL LEADERBOARD ---
                       Model  Best Val Loss (MSE)
3              roberta-large             0.718138
0          bert-base-uncased             0.901580
1            bert-base-cased             0.914746
2               roberta-base             1.021434
4  microsoft/deberta-v3-base             1.035869


### Step 4: Evaluate model performance on dev set

- Define helper function `get_prd`:
  - For **dev**: get both predictions and gold labels.
  - For **pred**: only get predictions (no gold labels).
- Define `evaluate_predictions_task1`:
  - Compute Pearson Correlation Coefficient (PCC) for Valence (V) and Arousal (A).
  - Compute normalized RMSE for combined VA score.
- Run evaluation on laptop and restaurant dev sets.
- Print metrics to check how well the models perform.


In [ ]:
import torch
import torch.nn as nn
import gc

# Define the winning model configuration
winner_model_name = "roberta-large"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model, optimizer, and loss
model = TransformerVARegressor(winner_model_name).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
loss_fn = nn.MSELoss()

# Get loaders for roberta-large
train_loader = all_loaders[winner_model_name]['train']
dev_loader = all_loaders[winner_model_name]['dev']

best_val_loss = float('inf')
print(f"Fine-tuning {winner_model_name} specifically for Laptop Eng...")

for epoch in range(4):
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss = eval_epoch(model, dev_loader, loss_fn, device)

    print(f"Epoch {epoch+1}/4 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_roberta_large_laptop.bin")
        print(" New best model saved!")

print("\n Fine-tuning complete.")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fine-tuning roberta-large specifically for Laptop Eng...
Epoch 1/4 | Train Loss: 2.3613 | Val Loss: 0.8073
⭐ New best model saved!
Epoch 2/4 | Train Loss: 0.7502 | Val Loss: 0.9129
Epoch 3/4 | Train Loss: 0.6500 | Val Loss: 0.8382
Epoch 4/4 | Train Loss: 0.5157 | Val Loss: 0.5846
⭐ New best model saved!

 Fine-tuning complete.


In [ ]:
# Load the best weights back in
model.load_state_dict(torch.load("best_roberta_large_laptop.bin"))

print("Calculating final metrics on Dev set...")

# 1. Get predictions and real labels
pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev", device=device)

# 2. Run the evaluation function
final_metrics = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)

print("\n--- LAPTOP DEV PERFORMANCE ---")
print(f"Valence Correlation (PCC_V): {final_metrics['PCC_Valence']}")
print(f"Arousal Correlation (PCC_A): {final_metrics['PCC_Arousal']}")
print(f"Combined Error (RMSE_VA):    {final_metrics['RMSE_VA']}")

Calculating final metrics on Dev set...

--- LAPTOP DEV PERFORMANCE ---
Valence Correlation (PCC_V): 0.9017000198364258
Arousal Correlation (PCC_A): 0.7404999732971191
Combined Error (RMSE_VA):    1.0813


In [ ]:
# The Restaurant English domain
subtask = "subtask_1"
lang = "eng"
domain = "restaurant"  # Transitioning from laptop to restaurant
task = "task1"

# Domain-specific URLs
train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl"
predict_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"

# Load the Restaurant data
train_raw_res = load_jsonl_url(train_url)
predict_raw_res = load_jsonl_url(predict_url)
train_df_res = jsonl_to_df(train_raw_res)
predict_df_res = jsonl_to_df(predict_raw_res)

# Split for validation
train_df_res, dev_df_res = train_test_split(train_df_res, test_size=0.1, random_state=42)

In [ ]:
# Initialize fresh model and loaders for the Restaurant domain
res_tokenizer = tokenizers["roberta-large"]
train_loader_res = DataLoader(MultiModelVADataset(train_df_res, res_tokenizer), batch_size=8, shuffle=True)
dev_loader_res = DataLoader(MultiModelVADataset(dev_df_res, res_tokenizer), batch_size=8)

model_res = TransformerVARegressor("roberta-large").to(device)
optimizer_res = torch.optim.AdamW(model_res.parameters(), lr=1e-5)

print("Starting training for Restaurant English...")
for epoch in range(4):
    train_loss = train_epoch(model_res, train_loader_res, optimizer_res, nn.MSELoss(), device)
    val_loss = eval_epoch(model_res, dev_loader_res, nn.MSELoss(), device)
    print(f"Epoch {epoch+1}: Val Loss = {val_loss:.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training for Restaurant English...
Epoch 1: Val Loss = 0.9502
Epoch 2: Val Loss = 0.8604
Epoch 3: Val Loss = 0.9442
Epoch 4: Val Loss = 0.5809


In [ ]:
# Evaluate the Restaurant model on the Laptop dev set
p_v, p_a, g_v, g_a = get_prd(model_res, all_loaders['roberta-large']['dev'], type="dev")
cross_domain_metrics = evaluate_predictions_task1(p_a, p_v, g_a, g_v)
print(f"Cross-Domain Performance (Res -> Laptop): {cross_domain_metrics}")

Cross-Domain Performance (Res -> Laptop): {'PCC_Arousal': np.float32(0.6966), 'PCC_Valence': np.float32(0.8973), 'RMSE_VA': np.float64(1.1525)}


In [ ]:
def sanitize_data(df):
    # Remove any rows that have missing (NaN) Valence or Arousal
    initial_len = len(df)
    df = df.dropna(subset=['Valence', 'Arousal'])

    # Ensure all scores are numbers
    df['Valence'] = pd.to_numeric(df['Valence'], errors='coerce')
    df['Arousal'] = pd.to_numeric(df['Arousal'], errors='coerce')
    df = df.dropna(subset=['Valence', 'Arousal'])

    #  Clip scores to 1-9 just in case
    df['Valence'] = df['Valence'].clip(1.0, 9.0)
    df['Arousal'] = df['Arousal'].clip(1.0, 9.0)

    print(f"Sanitized: Kept {len(df)} out of {initial_len} rows.")
    return df

# Apply it to your restaurant data
train_df_res = sanitize_data(train_df_res)
dev_df_res = sanitize_data(dev_df_res)

Sanitized: Kept 2796 out of 2796 rows.
Sanitized: Kept 311 out of 311 rows.


In [ ]:
# Use a very small batch size for the 'large' model
batch_size = 4

res_tokenizer = tokenizers["roberta-large"]

# Correctly create MultiModelVADataset instances first, then pass them to DataLoader
train_ds_res = MultiModelVADataset(train_df_res, res_tokenizer)
dev_ds_res = MultiModelVADataset(dev_df_res, res_tokenizer)

train_loader_res = DataLoader(train_ds_res, batch_size=batch_size, shuffle=True)
dev_loader_res = DataLoader(dev_ds_res, batch_size=batch_size)

# Initialize model on CPU first
model_res = TransformerVARegressor("roberta-large")

# Clear GPU memory and collect garbage before moving the model to GPU
torch.cuda.empty_cache()
gc.collect()

# Then move to device
model_res.to(device)

optimizer_res = torch.optim.AdamW(model_res.parameters(), lr=1e-5)
loss_fn = nn.MSELoss()

print("Starting a SAFE training run for Restaurant English...")

for epoch in range(4):
    model_res.train()
    total_loss = 0
    for batch in train_loader_res:
        optimizer_res.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model_res(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_res.parameters(), max_norm=1.0)

        optimizer_res.step()
        total_loss += loss.item()

    val_loss = eval_epoch(model_res, dev_loader_res, loss_fn, device)
    print(f"Epoch {epoch+1}: Train Loss: {total_loss/len(train_loader_res):.4f} | Val Loss: {val_loss:.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Step 5: Save and submit prediction results

- Define helper `df_to_jsonl`:
  - Sort by ID number.
  - Group rows by ID.
  - Save predictions in JSONL format (`ID`, `Aspect_VA`).
- Run the model on the predict sets (laptop & restaurant).
- Fill in predicted Valence/Arousal values.
- Export three JSONL files:




  - `pred_eng_laptop.jsonl`
  - `pred_eng_restaurant.jsonl`
  - `pred_zho_laptop.jsonl`
- These files can be uploaded as the final submission.


### File Naming Guidelines
When submitting your predictions on the Codabench task page:

Decide the target language(s) and domain(s). Each submission file corresponds to one language-domain combination.
For each language-domain combination, name the file pred_[lang_code]_[domain].jsonl, where
- [lang_code] represents a 3-letter language code, and
- [domain] represents a domain.
For example, Hausa predictions for the movie domain should be named pred_hau_movie.jsonl.
If submitting for multiple languages or domains, submit one prediction file per language-domain combination. For example, submitting for multiple languages or domains would look like this:
```plaintext
subtask_1
├── pred_eng_restaurant.jsonl
├── pred_eng_laptop.jsonl
└── pred_zho_laptop.jsonl

In [ ]:
#==== step 5 save & submit your predict results ====
def extract_num(s):
    m = re.search(r"(\d+)$", str(s))
    return int(m.group(1)) if m else -1

def df_to_jsonl(df, out_path):
    df_sorted = df.sort_values(by="ID", key=lambda x: x.map(extract_num))
    grouped = df_sorted.groupby("ID", sort=False)

    with open(out_path, "w", encoding="utf-8") as f:
        for gid, gdf in grouped:
            record = {
                "ID": gid,
                "Aspect_VA": []
            }
            for _, row in gdf.iterrows():
                record["Aspect_VA"].append({
                    "Aspect": row["Aspect"],
                    "VA": f"{row['Valence']:.2f}#{row['Arousal']:.2f}"
                })
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Re-loading the best laptop model to ensure a clean state for prediction
# Explicitly use the model saved for laptop fine-tuning from cell FQ5cGbf8BkP6
best_model_for_laptop_name = "roberta-large" # The architecture is roberta-large
best_model_for_laptop_path = "best_roberta_large_laptop.bin" # The file saved from laptop-specific fine-tuning

# Initialize model for prediction on CPU first
final_prediction_model = TransformerVARegressor(best_model_for_laptop_name)

# Then load state dict
final_prediction_model.load_state_dict(torch.load(best_model_for_laptop_path))

# Then move to device
final_prediction_model.to(device)
final_prediction_model.eval() # Set to evaluation mode

predict_tokenizer_for_submission = tokenizers[best_model_for_laptop_name] # Explicitly use the tokenizer for the best model
predict_df_for_submission = predict_df # This DataFrame holds the laptop prediction data
predict_lang_for_submission = "eng"
predict_domain_for_submission = "laptop"

pred_dataset = MultiModelVADataset(predict_df_for_submission, predict_tokenizer_for_submission)
pred_loader = DataLoader(pred_dataset, batch_size=64, shuffle=False) # Shuffle=False for prediction

# Get predictions, ensuring 'device' is passed
pred_v, pred_a = get_prd(final_prediction_model, pred_loader, type="pred", device=device)

predict_df_for_submission["Valence"] = pred_v
predict_df_for_submission["Arousal"] = pred_a

df_to_jsonl(predict_df_for_submission, f"pred_{predict_lang_for_submission}_{predict_domain_for_submission}.jsonl")

# Clean up to free memory after prediction
del final_prediction_model
torch.cuda.empty_cache()
import gc
gc.collect()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Download the submit files

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

# Create the folder subtask if it does not exist
os.makedirs(subtask, exist_ok=True)

# Move the three files into the subtask folder
for fname in [f"pred_{lang}_{domain}.jsonl"]:
    if os.path.exists(fname):
        shutil.move(fname, os.path.join(subtask, fname))

# Create a zip file named "submit.zip" containing the folder subtask
with zipfile.ZipFile(f"{subtask}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_in_dir in os.walk(subtask):
        for file in files_in_dir:
            path = os.path.join(root, file)
            # Keep folder structure inside the zip
            zf.write(path, os.path.relpath(path, "."))

# Download the created zip file to local machine
files.download(f"{subtask}.zip")

### Conclusion

In this notebook, we walked through the full pipeline for **Dimensional Aspect Sentiment Regression (DimASR)**:

1. **Load data**: Import the competition JSONL files, split train/dev sets, and convert to DataFrames.  
2. **Build dataset & dataloaders**: Define a custom `VADataset` to tokenize text and prepare `[Valence, Arousal]` labels.  
3. **Train & evaluate**: Train BERT-based regressors and check model performance on the dev sets using PCC and RMSE metrics.  
4. **Predict & submit**: Run the trained models on the prediction sets, generate VA scores, and save results as JSONL for submission.  

This pipeline ensures that your model is trained, validated, and ready for competition submission. You can further improve results by tuning hyperparameters, trying different pretrained models, or applying data augmentation strategies.
